# 24b — SDP Campaign Performance Validation v2

This notebook fixes and extends the SDP campaign-validation layer.

Key improvements over v1:

1. Calculates `sdp_vote_share_effective` from `sdp_votes / valid_votes` where raw `vote_share` is missing.
2. Attempts improved geography matching to WD25 using existing `WD25CD`, source ward codes, name matching, and optional OA21 source-year-to-WD25 crosswalks.
3. Allows provisional 2026 SDP-only rows to be included without forcing uncertain geography.
4. Keeps unmatched rows for review rather than silently dropping them.
5. Produces cleaner performance summaries by tribe, latest top party, region, ward, and year.

Outputs are written to:

```text
data/processed/sdp_validation_v2/
```

In [1]:
from pathlib import Path
from datetime import datetime
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 240)

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
GEOGRAPHY_DIR = DATA_DIR / "geography"
RAW_DIR = DATA_DIR / "raw"

INPUT_DIRS = [
    PROCESSED_DIR / "sdp_validation_v2",
    PROCESSED_DIR / "sdp_validation_v1",
    PROCESSED_DIR / "target_review_pack_v1",
    PROCESSED_DIR / "caveat_resolution_v2",
    PROCESSED_DIR / "target_model_v2",
    PROCESSED_DIR / "target_model_v1" / "inputs",
    PROCESSED_DIR / "aggregations_v1",
    PROCESSED_DIR / "election_results",
    PROCESSED_DIR,
    RAW_DIR / "election_results",
    RAW_DIR,
]

OUTPUT_DIR = PROCESSED_DIR / "sdp_validation_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Main optional inputs. Change filenames here if your local names differ.
EXISTING_SDP_PROFILE_FILENAME = "sdp_campaign_wards_profile_v1.csv"
LOCAL_ELECTION_RAW_FILENAME = "local_election_results_raw_v1.csv"
PROVISIONAL_2026_FILENAME = "sdp_candidate_results_2026_raw_v1.csv"
SDP_WORKBOOK_FILENAME = "SDP Election Results since 2018.xlsx"
MODEL_REVIEW_FILENAME = "all_available_consolidated_target_review_v1.csv"
NW_MODEL_REVIEW_FILENAME = "north_west_revised_consolidated_review_v2.csv"
ATLAS_FILENAME = "k7_ward25_atlas_with_latest_election_metrics_v1.csv"
WARD_METRICS_YEAR_FILENAME = "k7_ward25_election_metrics_by_source_year_v1.csv"

EXPECTED_COUNTS = {2021: 68, 2022: 30, 2023: 36, 2024: 28, 2025: 11, 2026: 48}

print("Project:", PROJECT_DIR)
print("Output:", OUTPUT_DIR)

Project: c:\Users\keena\Documents\Electoral_Tribes
Output: c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_validation_v2


In [2]:
def find_file(filename, required=False):
    for folder in INPUT_DIRS + [GEOGRAPHY_DIR, PROJECT_DIR, Path("/mnt/data")]:
        p = folder / filename
        if p.exists():
            return p
    for root in [PROCESSED_DIR, RAW_DIR, GEOGRAPHY_DIR, PROJECT_DIR]:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return sorted(matches, key=lambda x: x.stat().st_mtime, reverse=True)[0]
    if required:
        raise FileNotFoundError(filename)
    return None


def read_csv_optional(filename):
    p = find_file(filename, required=False)
    if p is None:
        print("Optional CSV missing:", filename)
        return None, None
    df = pd.read_csv(p, low_memory=False)
    print(f"Loaded {filename}: {df.shape} from {p}")
    return df, p


def clean_text(x):
    if pd.isna(x): return ""
    x = str(x).lower().strip().replace("&", " and ")
    x = re.sub(r"st[.]?", "saint", x)
    x = re.sub(r"[^a-z0-9]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()


def norm_code(x):
    if pd.isna(x): return np.nan
    x = str(x).strip()
    return x if x and x.lower() not in ["nan", "none", "null"] else np.nan


def to_num(s):
    return pd.to_numeric(s, errors="coerce")


def standardise_party(x):
    if pd.isna(x): return ""
    x = str(x).strip().lower()
    x = re.sub(r"[^a-z0-9]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def is_sdp_party(series):
    s = series.fillna("").astype(str).str.lower()
    # Include SDP / Social Democratic Party, exclude SDLP and unrelated strings.
    positive = s.str.contains(r"sdp|social democratic party", regex=True)
    negative = s.str.contains(r"sdlp|social democratic labour", regex=True)
    return positive & ~negative

## 24b.1 Load model/atlas files

The preferred model file is the all-available consolidated review table. If unavailable, the notebook falls back to the K7 ward atlas.

In [3]:
model, model_path = read_csv_optional(MODEL_REVIEW_FILENAME)
if model is None:
    model, model_path = read_csv_optional(ATLAS_FILENAME)

if model is None:
    raise FileNotFoundError("No model/atlas file found. Need all_available_consolidated_target_review_v1.csv or k7_ward25_atlas_with_latest_election_metrics_v1.csv")

model["WD25CD"] = model["WD25CD"].astype(str).str.strip()

# Optional North West revised caveat file can enrich report confidence if present.
nw_revised, _ = read_csv_optional(NW_MODEL_REVIEW_FILENAME)
if nw_revised is not None and "WD25CD" in nw_revised.columns:
    nw_revised["WD25CD"] = nw_revised["WD25CD"].astype(str).str.strip()
    enrich_cols = [c for c in ["WD25CD", "report_confidence_band_v2", "revised_strategic_lane_v2", "report_caveat_summary_v2"] if c in nw_revised.columns]
    model = model.merge(nw_revised[enrich_cols].drop_duplicates("WD25CD"), on="WD25CD", how="left", validate="many_to_one")

print("Model rows:", len(model))

Loaded all_available_consolidated_target_review_v1.csv: (7572, 47) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1\all_available_consolidated_target_review_v1.csv
Loaded north_west_revised_consolidated_review_v2.csv: (825, 74) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\caveat_resolution_v2\north_west_revised_consolidated_review_v2.csv
Model rows: 7572


## 24b.2 Load SDP candidate rows

Load order:

1. Existing `sdp_campaign_wards_profile_v1.csv` if present.
2. Raw candidate-level election file if present.
3. Provisional 2026 SDP file if present.
4. Optional workbook import if present.

The notebook will deduplicate on the best available identifiers.

In [4]:
frames = []
source_notes = []

# 1. Existing v1 profile.
existing, existing_path = read_csv_optional(EXISTING_SDP_PROFILE_FILENAME)
if existing is not None:
    existing["source_mode"] = existing.get("source_mode", "existing_sdp_profile_v1")
    frames.append(existing)
    source_notes.append(("existing_sdp_profile_v1", len(existing), str(existing_path)))

# 2. Raw candidate-level file.
raw, raw_path = read_csv_optional(LOCAL_ELECTION_RAW_FILENAME)
if raw is not None and "party_label" in raw.columns:
    sdp_raw = raw[is_sdp_party(raw["party_label"])].copy()
    sdp_raw["source_mode"] = "local_election_results_raw_extracted"
    frames.append(sdp_raw)
    source_notes.append(("local_election_results_raw_extracted", len(sdp_raw), str(raw_path)))

# 3. Provisional 2026 file.
prov26, prov26_path = read_csv_optional(PROVISIONAL_2026_FILENAME)
if prov26 is not None:
    prov26["source_year"] = prov26.get("source_year", prov26.get("election_year", 2026))
    prov26["election_year"] = prov26.get("election_year", prov26["source_year"])
    prov26["source_mode"] = "provisional_2026_sdp_file"
    frames.append(prov26)
    source_notes.append(("provisional_2026_sdp_file", len(prov26), str(prov26_path)))

# 4. Optional workbook import. This is permissive and may require local column cleanup.
workbook_path = find_file(SDP_WORKBOOK_FILENAME, required=False)
if workbook_path is not None:
    try:
        xls = pd.ExcelFile(workbook_path)
        workbook_frames = []
        for sheet in xls.sheet_names:
            temp = pd.read_excel(workbook_path, sheet_name=sheet)
            if len(temp) == 0:
                continue
            # Only keep sheets that appear to contain candidate/result rows.
            col_clean = {c: clean_text(c) for c in temp.columns}
            temp = temp.rename(columns={c: col_clean[c] for c in temp.columns})
            if any("candidate" in c for c in temp.columns) or any("sdp" in c for c in temp.columns) or any("votes" in c for c in temp.columns):
                temp["source_sheet"] = sheet
                temp["source_mode"] = "sdp_workbook_import"
                workbook_frames.append(temp)
        if workbook_frames:
            wb = pd.concat(workbook_frames, ignore_index=True, sort=False)
            frames.append(wb)
            source_notes.append(("sdp_workbook_import", len(wb), str(workbook_path)))
    except Exception as e:
        print("Workbook import failed; continuing without it:", e)

if not frames:
    raise FileNotFoundError("No SDP candidate/result data found. Provide sdp_candidate_results_v1.csv, local_election_results_raw_v1.csv, existing v1 output, workbook, or provisional 2026 file.")

sdp = pd.concat(frames, ignore_index=True, sort=False)
print("Combined SDP candidate/result rows before standardisation:", len(sdp))
display(pd.DataFrame(source_notes, columns=["source", "rows", "path"]))

Loaded sdp_campaign_wards_profile_v1.csv: (171, 76) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v1\sdp_campaign_wards_profile_v1.csv
Loaded local_election_results_raw_v1.csv: (80392, 52) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\local_election_results_raw_v1.csv
Optional CSV missing: sdp_candidate_results_2026_raw_v1.csv
Combined SDP candidate/result rows before standardisation: 171


,source,rows,path
0,existing_sdp_profile_v1,171,c:\Users\keena\Documents\Electoral_Tribes\data...


## 24b.3 Standardise SDP result columns and calculate vote share

In [5]:
# Column aliases for permissive loading.
def first_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# Normalise commonly expected columns.
aliases = {
    "election_year": ["election_year", "source_year", "year"],
    "election_date": ["election_date", "date"],
    "council_name": ["council_name", "council", "local_authority", "lad_name"],
    "ward_name": ["ward_name", "ward", "division_name"],
    "candidate_name": ["candidate_name", "candidate", "name"],
    "party_label": ["party_label", "party", "description", "standard_party_label"],
    "sdp_votes": ["sdp_votes", "votes", "votes_effective", "candidate_votes"],
    "valid_votes": ["valid_votes", "candidate_effective_total_votes", "total_votes"],
    "vote_share": ["sdp_vote_share", "vote_share", "share"],
    "sdp_position": ["sdp_position", "rank", "position"],
    "winner_party": ["winner_party", "top_party_by_votes", "top_party_bucket"],
    "runner_up_party": ["runner_up_party", "runner_up_party_by_votes", "runner_up_party_bucket"],
    "WD25CD": ["WD25CD", "matched_wd25cd", "ward25cd", "wd25cd"],
    "WD25NM": ["WD25NM", "matched_wd25nm", "ward25nm", "wd25nm"],
    "LAD25CD": ["LAD25CD", "matched_lad25cd", "lad25cd"],
    "LAD25NM": ["LAD25NM", "matched_lad25nm", "lad25nm"],
    "ward_code": ["ward_code", "source_geography_code", "ec_ward_code"],
    "result_area_key": ["result_area_key"],
}

std = pd.DataFrame(index=sdp.index)
for standard, candidates in aliases.items():
    c = first_col(sdp, candidates)
    std[standard] = sdp[c] if c is not None else np.nan

# Retain useful source columns if present.
for c in ["source_mode", "source_file", "source_sheet", "source_url", "source_notes", "mapping_confidence", "mapping_notes", "boundary_year", "result_id"]:
    std[c] = sdp[c] if c in sdp.columns else np.nan

# Candidate filter if importing non-SDP rows from a workbook.
if "party_label" in std.columns:
    party_s = std["party_label"].fillna("").astype(str)
    if party_s.str.len().gt(0).any():
        keep = is_sdp_party(party_s) | std["source_mode"].eq("existing_sdp_profile_v1") | std["source_mode"].eq("provisional_2026_sdp_file") | std["source_mode"].eq("sdp_workbook_import")
        std = std[keep].copy()

std["election_year"] = to_num(std["election_year"]).astype("Int64")
std["source_year"] = std["election_year"]
std["sdp_votes"] = to_num(std["sdp_votes"])
std["valid_votes"] = to_num(std["valid_votes"])

# Clean vote_share and compute effective vote share.
raw_share = to_num(std["vote_share"])
# If a source stores 14.2 rather than 0.142, convert.
raw_share = np.where(raw_share > 1, raw_share / 100, raw_share)
std["sdp_vote_share_raw_clean"] = pd.Series(raw_share, index=std.index)
std["sdp_vote_share_from_votes"] = np.where(std["valid_votes"] > 0, std["sdp_votes"] / std["valid_votes"], np.nan)
std["sdp_vote_share_effective"] = std["sdp_vote_share_raw_clean"].combine_first(pd.Series(std["sdp_vote_share_from_votes"], index=std.index))

# Standard matching helper fields.
std["WD25CD"] = std["WD25CD"].map(norm_code)
std["ward_code"] = std["ward_code"].map(norm_code)
std["clean_council_name"] = std["council_name"].map(clean_text)
std["clean_ward_name"] = std["ward_name"].map(clean_text)

print("Standardised SDP rows:", len(std))
display(std.head())

Standardised SDP rows: 11


,election_year,election_date,council_name,ward_name,candidate_name,party_label,sdp_votes,valid_votes,vote_share,sdp_position,winner_party,runner_up_party,WD25CD,WD25NM,LAD25CD,LAD25NM,ward_code,result_area_key,source_mode,source_file,source_sheet,source_url,source_notes,mapping_confidence,mapping_notes,boundary_year,result_id,source_year,sdp_vote_share_raw_clean,sdp_vote_share_from_votes,sdp_vote_share_effective,clean_council_name,clean_ward_name
160,2025,2025-05-01,Buckinghamshire,Haddenham & Stone,George Millo,Social Democratic Party,85,NaN,NaN,NaN,NaN,NaN,E05015954,Haddenham & Stone,NaN,Buckinghamshire,E05015954,2025|CODE|BUCKINGHAMSHIRE|E05015954|HADDENHAM_...,auto_extracted_candidate_database,LEH-2025-results-HoC.xlsx,Candidates result,NaN,House of Commons Library Local Election Handbo...,NaN,NaN,2025,ed348ebdd491a571,2025,NaN,NaN,NaN,buckinghamshire,haddenham and stone
161,2025,2025-05-01,Buckinghamshire,Ridgeway East,Yvonne Dorette Wilding,Social Democratic Party,154,NaN,NaN,NaN,NaN,NaN,E05015966,Ridgeway East,NaN,Buckinghamshire,E05015966,2025|CODE|BUCKINGHAMSHIRE|E05015966|RIDGEWAY_EAST,auto_extracted_candidate_database,LEH-2025-results-HoC.xlsx,Candidates result,NaN,House of Commons Library Local Election Handbo...,NaN,NaN,2025,c991c164b20cd8fe,2025,NaN,NaN,NaN,buckinghamshire,ridgeway east
162,2025,2025-05-01,Buckinghamshire,Totteridge & Bowerdean,Adam Williams,Social Democratic Party,34,NaN,NaN,NaN,NaN,NaN,E05015971,Totteridge & Bowerdean,NaN,Buckinghamshire,E05015971,2025|CODE|BUCKINGHAMSHIRE|E05015971|TOTTERIDGE...,auto_extracted_candidate_database,LEH-2025-results-HoC.xlsx,Candidates result,NaN,House of Commons Library Local Election Handbo...,NaN,NaN,2025,78ca04933ba6d64f,2025,NaN,NaN,NaN,buckinghamshire,totteridge and bowerdean
163,2025,2025-05-01,Durham,Chester-le-Street North,Tom Chittenden,Social Democratic Party,83,NaN,NaN,NaN,NaN,NaN,E05016163,Chester-le-Street North,NaN,County Durham,E05016163,2025|CODE|DURHAM|E05016163|CHESTER_LE_STREET_N...,auto_extracted_candidate_database,LEH-2025-results-HoC.xlsx,Candidates result,NaN,House of Commons Library Local Election Handbo...,NaN,NaN,2025,86622250b13f41a1,2025,NaN,NaN,NaN,durham,chester le street north
164,2025,2025-05-01,South Kesteven,Folkingham Rural,Alexander Leslie Mitchell,Social Democratic Party,50,NaN,NaN,NaN,NaN,NaN,E58000906,NaN,NaN,NaN,E58000906,2025|CODE|SOUTH_KESTEVEN|E58000906|FOLKINGHAM_...,auto_extracted_candidate_database,LEH-2025-results-HoC.xlsx,Candidates result,NaN,House of Commons Library Local Election Handbo...,NaN,NaN,2025,fb8a798c2a84cb5e,2025,NaN,NaN,NaN,south kesteven,folkingham rural


## 24b.4 Build optional source-year ward-to-WD25 crosswalks

If OA21 lookup files are available, the notebook will map source-year ward codes to WD25 codes using the modal WD25 ward by OA count. This is especially useful for 2022, 2023 and 2024 source-year ward codes.

Required optional files, if available:

```text
oa21_to_wd22_lad22_ctyua22_rgn22_ctry22_eng_wal.csv
oa21_to_wd23_lad23_eng_wal.csv
oa21_to_wd24_lad24_eng_wal.csv
oa21_to_wd25_lad25_eng_wal(may25).csv
```

In [6]:
def find_lookup_for_year(year):
    patterns = [
        f"*wd{str(year)[-2:]}*lad{str(year)[-2:]}*.csv",
        f"*WD{str(year)[-2:]}*LAD{str(year)[-2:]}*.csv",
        f"oa21_to_wd{str(year)[-2:]}*.csv",
    ]
    for root in [GEOGRAPHY_DIR, PROCESSED_DIR, PROJECT_DIR, Path("/mnt/data")]:
        if not root.exists():
            continue
        for pat in patterns:
            matches = list(root.rglob(pat))
            if matches:
                return sorted(matches, key=lambda x: x.stat().st_mtime, reverse=True)[0]
    return None


def standardise_lookup(df, year):
    # Identify columns flexibly.
    lower = {c.lower(): c for c in df.columns}
    def pick(cands):
        for cand in cands:
            if cand.lower() in lower: return lower[cand.lower()]
        # fuzzy contains fallback
        for c in df.columns:
            lc = c.lower()
            for cand in cands:
                if cand.lower() in lc: return c
        return None
    yy = str(year)[-2:]
    oa = pick(["OA21CD", "oa21cd"])
    wd = pick([f"WD{yy}CD", f"wd{yy}cd", "WDCD", "ward_code"])
    wdnm = pick([f"WD{yy}NM", f"wd{yy}nm", "WDNM", "ward_name"])
    lad = pick([f"LAD{yy}CD", f"lad{yy}cd", "LADCD"])
    ladnm = pick([f"LAD{yy}NM", f"lad{yy}nm", "LADNM"])
    required = [oa, wd]
    if any(x is None for x in required):
        return None
    out = df[[oa, wd] + ([wdnm] if wdnm else []) + ([lad] if lad else []) + ([ladnm] if ladnm else [])].copy()
    rename = {oa: "OA21CD", wd: "source_ward_code"}
    if wdnm: rename[wdnm] = "source_ward_name"
    if lad: rename[lad] = "source_lad_code"
    if ladnm: rename[ladnm] = "source_lad_name"
    out = out.rename(columns=rename)
    out["source_year"] = year
    return out.drop_duplicates()

# Load 2025 lookup as target side.
wd25_path = find_lookup_for_year(2025)
wd25_lookup = None
if wd25_path:
    tmp = pd.read_csv(wd25_path, low_memory=False)
    wd25_lookup = standardise_lookup(tmp, 2025)
    if wd25_lookup is not None:
        wd25_lookup = wd25_lookup.rename(columns={
            "source_ward_code": "WD25CD",
            "source_ward_name": "WD25NM",
            "source_lad_code": "LAD25CD",
            "source_lad_name": "LAD25NM",
        })
        wd25_lookup = wd25_lookup[[c for c in ["OA21CD", "WD25CD", "WD25NM", "LAD25CD", "LAD25NM"] if c in wd25_lookup.columns]].drop_duplicates()
        print("Loaded WD25 lookup:", wd25_path, wd25_lookup.shape)
else:
    print("No WD25 OA lookup found. Source-year crosswalks disabled.")

crosswalks = []
if wd25_lookup is not None:
    for year in [2022, 2023, 2024, 2025]:
        path = find_lookup_for_year(year)
        if path is None:
            print("No source-year lookup found for", year)
            continue
        src = standardise_lookup(pd.read_csv(path, low_memory=False), year)
        if src is None:
            print("Could not standardise lookup for", year, path)
            continue
        joined = src.merge(wd25_lookup, on="OA21CD", how="inner")
        if len(joined) == 0:
            continue
        group_cols = ["source_year", "source_ward_code", "WD25CD"]
        counts = joined.groupby(group_cols, as_index=False).agg(oa_count=("OA21CD", "nunique"))
        totals = counts.groupby(["source_year", "source_ward_code"])["oa_count"].transform("sum")
        counts["wd25_match_share"] = counts["oa_count"] / totals
        idx = counts.sort_values(["source_year", "source_ward_code", "oa_count"], ascending=[True, True, False]).groupby(["source_year", "source_ward_code"]).head(1).index
        best = counts.loc[idx].copy()
        # Add names.
        best = best.merge(joined[["source_year", "source_ward_code", "WD25CD"] + [c for c in ["WD25NM", "LAD25CD", "LAD25NM"] if c in joined.columns]].drop_duplicates(), on=["source_year", "source_ward_code", "WD25CD"], how="left")
        crosswalks.append(best)
        print("Built source-year -> WD25 crosswalk", year, best.shape)

source_to_wd25 = pd.concat(crosswalks, ignore_index=True, sort=False) if crosswalks else pd.DataFrame()
if len(source_to_wd25):
    source_to_wd25.to_csv(OUTPUT_DIR / "source_ward_to_wd25_crosswalk_v2.csv", index=False)
    display(source_to_wd25.head())
else:
    print("No source-year crosswalks created.")

No source-year crosswalks created.


## 24b.5 Match SDP rows to WD25 model rows

In [7]:

matched = std.copy()
matched["match_method_v2"] = "unmatched"
matched["match_confidence_v2"] = "unmatched"
matched["WD25CD_final"] = matched["WD25CD"].map(norm_code)
matched["wd25_match_share_v2"] = np.nan

# 1. Already has WD25CD.
mask = matched["WD25CD_final"].notna()
matched.loc[mask, "match_method_v2"] = "existing_wd25cd"
matched.loc[mask, "match_confidence_v2"] = "high"

# 2. Source ward code -> WD25 crosswalk.
if len(source_to_wd25):
    to_match = matched[matched["WD25CD_final"].isna()].copy()
    to_match["_orig_index"] = to_match.index
    to_match["source_year"] = to_num(to_match["source_year"])
    bridge = source_to_wd25[["source_year", "source_ward_code", "WD25CD", "WD25NM", "LAD25CD", "LAD25NM", "wd25_match_share"]].drop_duplicates()
    temp = to_match.merge(
        bridge,
        left_on=["source_year", "ward_code"],
        right_on=["source_year", "source_ward_code"],
        how="left"
    )
    use = temp["WD25CD_y" if "WD25CD_y" in temp.columns else "WD25CD"].notna()
    if use.any():
        wd_col = "WD25CD_y" if "WD25CD_y" in temp.columns else "WD25CD"
        orig_idx = temp.loc[use, "_orig_index"].values
        matched.loc[orig_idx, "WD25CD_final"] = temp.loc[use, wd_col].values
        matched.loc[orig_idx, "match_method_v2"] = "source_ward_to_wd25_crosswalk"
        matched.loc[orig_idx, "match_confidence_v2"] = np.where(temp.loc[use, "wd25_match_share"].ge(0.95), "high", "medium")
        matched.loc[orig_idx, "wd25_match_share_v2"] = temp.loc[use, "wd25_match_share"].values

# 3. Exact council+ward name match to model where still unmatched.
model_names = model[[c for c in ["WD25CD", "WD25NM", "LAD25NM"] if c in model.columns]].drop_duplicates().copy()
model_names["clean_ward_name"] = model_names.get("WD25NM", pd.Series("", index=model_names.index)).map(clean_text)
model_names["clean_council_name"] = model_names.get("LAD25NM", pd.Series("", index=model_names.index)).map(clean_text)
name_map = model_names.drop_duplicates(["clean_council_name", "clean_ward_name"])

still = matched[matched["WD25CD_final"].isna()].copy()
if len(still):
    still["_orig_index"] = still.index
    temp = still.merge(
        name_map[["clean_council_name", "clean_ward_name", "WD25CD"]],
        on=["clean_council_name", "clean_ward_name"],
        how="left"
    )
    use = temp["WD25CD"].notna()
    if use.any():
        orig_idx = temp.loc[use, "_orig_index"].values
        matched.loc[orig_idx, "WD25CD_final"] = temp.loc[use, "WD25CD"].values
        matched.loc[orig_idx, "match_method_v2"] = "exact_council_ward_name_to_wd25"
        matched.loc[orig_idx, "match_confidence_v2"] = "medium"

# Merge model fields. Rename right-side WD25CD so the final source key stays unambiguous.
model_cols = [c for c in [
    "WD25CD", "WD25NM", "LAD25CD", "LAD25NM", "analysis_region", "strategic_lane", "revised_strategic_lane_v2", "report_confidence_band_v2", "report_caveat_summary_v2",
    "initial_watchlist_score", "demographic_relevance_score", "electoral_opportunity_score", "political_openness_score", "breakthrough_complacency_score",
    "dominant_cluster_name", "second_cluster_name", "latest_election_top_party_bucket", "latest_election_runner_up_party_bucket"
] if c in model.columns]
model_enrich = model[model_cols].drop_duplicates("WD25CD").rename(columns={"WD25CD": "WD25CD_model"})

matched = matched.merge(model_enrich, left_on="WD25CD_final", right_on="WD25CD_model", how="left", validate="many_to_one")
matched["matched_to_model_v2"] = matched["WD25CD_model"].notna()

# Harmonise final model columns. Keep original source WD25CD as source_WD25CD, and make WD25CD the final matched model key.
matched = matched.rename(columns={"WD25CD": "source_WD25CD"})
matched["WD25CD"] = matched["WD25CD_model"].combine_first(matched["WD25CD_final"])

print("Match methods:")
display(matched["match_method_v2"].value_counts(dropna=False).reset_index(name="rows"))
print("Matched to model:", matched["matched_to_model_v2"].sum(), "of", len(matched))


Match methods:


,match_method_v2,rows
0,existing_wd25cd,11


Matched to model: 7 of 11


## 24b.6 Candidate-count QA and performance summaries

In [8]:
# Candidate count check against user-provided expected counts.
observed = matched.groupby("election_year", dropna=False).size().reset_index(name="observed_sdp_candidate_rows")
expected = pd.DataFrame([{"election_year": y, "expected_user_provided": n} for y, n in EXPECTED_COUNTS.items()])
count_check = expected.merge(observed, on="election_year", how="left")
count_check["observed_sdp_candidate_rows"] = count_check["observed_sdp_candidate_rows"].fillna(0).astype(int)
count_check["difference_observed_minus_expected"] = count_check["observed_sdp_candidate_rows"] - count_check["expected_user_provided"]

# Summaries.
def perf_summary(group_cols, filename):
    cols = [c for c in group_cols if c in matched.columns]
    if not cols:
        return pd.DataFrame()
    out = (
        matched.groupby(cols, dropna=False, as_index=False)
        .agg(
            sdp_candidate_rows=("candidate_name", "size"),
            matched_rows=("matched_to_model_v2", "sum"),
            total_sdp_votes=("sdp_votes", "sum"),
            mean_sdp_vote_share=("sdp_vote_share_effective", "mean"),
            median_sdp_vote_share=("sdp_vote_share_effective", "median"),
            max_sdp_vote_share=("sdp_vote_share_effective", "max"),
            mean_model_score=("initial_watchlist_score", "mean"),
        )
        .sort_values(["max_sdp_vote_share", "total_sdp_votes"], ascending=False)
    )
    out.to_csv(OUTPUT_DIR / filename, index=False)
    return out

by_tribe = perf_summary(["dominant_cluster_name"], "sdp_performance_by_dominant_tribe_v2.csv")
by_party = perf_summary(["latest_election_top_party_bucket"], "sdp_performance_by_latest_top_party_v2.csv")
by_region = perf_summary(["analysis_region"], "sdp_performance_by_region_v2.csv")
by_year = perf_summary(["election_year"], "sdp_performance_by_year_v2.csv")
by_ward = perf_summary(["LAD25NM", "WD25NM", "WD25CD"], "sdp_repeat_campaign_wards_v2.csv")

# Highest vote-share cases.
highest = matched.sort_values(["sdp_vote_share_effective", "sdp_votes"], ascending=False)
highest.to_csv(OUTPUT_DIR / "sdp_highest_vote_share_cases_v2.csv", index=False)

# Correlation file.
score_cols = ["initial_watchlist_score", "demographic_relevance_score", "electoral_opportunity_score", "political_openness_score", "breakthrough_complacency_score"]
cor_rows = []
for col in score_cols:
    if col in matched.columns:
        sub = matched[[col, "sdp_vote_share_effective"]].dropna()
        if len(sub) >= 3:
            cor_rows.append({"score_column": col, "n": len(sub), "corr_with_sdp_vote_share": sub[col].corr(sub["sdp_vote_share_effective"])})
correlation = pd.DataFrame(cor_rows)

# Unmatched review.
unmatched = matched[~matched["matched_to_model_v2"]].copy()

# Save main files.
matched.to_csv(OUTPUT_DIR / "sdp_campaign_wards_profile_v2.csv", index=False)
unmatched.to_csv(OUTPUT_DIR / "sdp_unmatched_results_review_v2.csv", index=False)
count_check.to_csv(OUTPUT_DIR / "sdp_candidate_count_check_v2.csv", index=False)
correlation.to_csv(OUTPUT_DIR / "sdp_campaign_vs_model_score_correlation_v2.csv", index=False)

print("Candidate count check")
display(count_check)
print("Matched rows:", matched["matched_to_model_v2"].sum(), "Unmatched:", len(unmatched))
display(correlation)

Candidate count check


,election_year,expected_user_provided,observed_sdp_candidate_rows,difference_observed_minus_expected
0,2021,68,0,-68
1,2022,30,0,-30
2,2023,36,0,-36
3,2024,28,0,-28
4,2025,11,11,0
5,2026,48,0,-48


Matched rows: 7 Unmatched: 4


""
